In [ ]:
# ====================== GPU PICKER (server-friendly) ======================
# Set which GPU to use (0, 1, ...). Set to None to NOT force anything (respects the environment).
GPU_ID = 1 # Change to None when you want to leave it free for other users
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
torch.backends.cudnn.benchmark = True  # if the size varies A LOT, consider False
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.set_device(0)
print("Device:", device, "| Visible devices (after mapping):", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if device == "cuda":
    try:
        print("Using mapped GPU 0:", torch.cuda.get_device_name(0))
    except Exception:
        pass
    for i in range(torch.cuda.device_count()):
        try:
            print(f"Device {i}:", torch.cuda.get_device_name(i))
        except Exception:
            pass
import sys
import time
from datetime import datetime

In [ ]:
import sys
sys.path.insert(0, "/workspace/app")
import matplotlib.pyplot as plt
import pandas as pd
import random
import shap
import numpy as np
import os
import ast
import pickle
from coding.Data_Processing import Run_RNN as rf
import importlib
importlib.reload(rf)
from coding.Data_Preprocessing import DataPreprocessing_melt as mf
from coding.Data_Processing import TargetVariables as tv
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
from torch.cuda.amp import autocast
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90  

___

LOG

In [ ]:
class Tee(object):
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

In [ ]:
log_path = f"BASE_REG_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
orig_stdout = sys.stdout
orig_stderr = sys.stderr
log_file = open(log_path, "w", buffering=1) 
sys.stdout = Tee(orig_stdout, log_file)
sys.stderr = Tee(orig_stderr, log_file)
start_time = time.perf_counter()
print("---- Logging started ----")
print("Log file:", log_path)

___

Dataset

In [ ]:
df_wav = pd.read_pickle('/workspace/app/planilhas/df_emb3D.pkl') # wav2vec
df_wav['Patient_ID'] = encoder.fit_transform(df_wav['Patient_ID'])
print(f"Patients: {df_wav['Patient_ID'].nunique()}")

In [ ]:
df_wav = tv.standardized_T1(df_wav, MAX_HDRS, MAX_CDI)
df_wav["Y_Standardized_T1"] = pd.to_numeric(df_wav["Y_Standardized_T1"], errors="coerce")

___

Preprocessing

In [ ]:
suffixes = ('Date', 'F1_Score', 'F2_Score', 'F3_Score', '_Days', 'soc_Score', 'iso_Score', 'sup_Score')
prefixes = ('Questionnary1', 'Questionnary2', 'LogMel_Pat_Speech', 'Days_')
exact_cols = {
    "Age", 'Patient_Gender', 'Randomization_Group', 'Education', 'Number_Complete_Sessions',
    'Questionnary3_TAS_20_T0_TOT_Score', 'Questionnary4_AQC_T0_TOT_Score', 'Questionnary5_HQ_25_T0_TOT_Score',
    'Questionnary3_TAS_20_T1_TOT_Score', 'Questionnary4_AQC_T1_TOT_Score', 'Questionnary5_HQ_25_T1_TOT_Score'}
cols_to_drop = [
    col for col in df_wav.columns 
    if col in exact_cols or col.endswith(suffixes) or col.startswith(prefixes)]
df_wav.drop(columns=cols_to_drop, inplace=True)

In [ ]:
meta = ['Patient_ID', 'Y_Standardized_T1']
df_wav = mf.get_embeddings_per_segment_mean_3D(df_wav, meta)

In [ ]:
df_cnn = pd.read_pickle("/workspace/app/planilhas/df_cnn_segments_reg.pkl")
df_cnn = df_cnn.drop(columns=["Questionnary3_TAS_20_T0_TOT_Score","Questionnary4_AQC_T0_TOT_Score"])

___

### Experiments

In [ ]:
unique_patients = df_cnn["Patient_ID"].unique()
target = "Y_Standardized_T1"
num_epochs = 20
BS = 2

#### Experiment 1 - 
a) Baseline Regression using CNN-log 

In [ ]:
representation = "cnn"
npy_col = 'CNN_SegEmb_npy'
experiments = [
    {"name": "GRU_LogMel", "model_name": "GRU", "kwargs": {}},
    {"name": "TRANS_LogMel", "model_name": "TRANS", "kwargs": {"pooling_trans": "mean"}}
]
for exp in experiments:
    print(f"Processing Dataset {exp['name']} Regression")
    results = [rf.reg_lopoRNN(p, df_cnn, 'Patient_ID', npy_col, target, exp["model_name"],
            representation=representation, num_epochs=num_epochs, BS=BS, **exp["kwargs"]
        )for p in unique_patients]
    print(f"{exp['name']} Done")
    df_results = pd.DataFrame(results)
    df_summary = pd.DataFrame([{
        "Model": exp["name"],
        "RMSE_Mean": df_results["RMSE"].mean(),
        "RMSE_Std": df_results["RMSE"].std(),
        "MSE_Mean": df_results["MSE"].mean(),
        "MAE_Mean": df_results["MAE"].mean()
    }])
    df_results.to_excel(f'/workspace/app/planilhas/TaCoLa/REG_{exp["name"]}.xlsx', index=False)
    #df_summary.to_excel(f'/workspace/app/planilhas/TaCoLa/REG_{exp["name"]}_SUMMARY.xlsx', index=False)

b) Baseline Regression using wav2vec

In [ ]:
representation = "wav2vec"
npy_col = "Embedding_npy"
experiments = [
    {"name": "GRU_Wav", "model_name": "GRU", "kwargs": {}},
    {"name": "TRANS_Wav","model_name": "TRANS","kwargs": {"pooling_trans": "mean"}}]
for exp in experiments:
    print(f"Processing Dataset {exp['name']} Regression")
    results = [rf.reg_lopoRNN(p, df_wav, 'Patient_ID', npy_col, target, exp["model_name"],
            representation=representation, num_epochs=num_epochs, BS=BS, **exp["kwargs"]
        )for p in unique_patients]
    print(f"{exp['name']} Done")
    df_results = pd.DataFrame(results)
    df_summary = pd.DataFrame([{
        "Model": exp["name"],
        "RMSE_Mean": df_results["RMSE"].mean(),
        "RMSE_Std": df_results["RMSE"].std(),
        "MSE_Mean": df_results["MSE"].mean(),
        "MAE_Mean": df_results["MAE"].mean()
    }])
    df_results.to_excel(f'/workspace/app/planilhas/TaCoLa/REG_{exp["name"]}.xlsx', index=False)
    #df_summary.to_excel(f'/workspace/app/planilhas/TaCoLa/REG_{exp["name"]}_SUMMARY.xlsx', index=False)

Seeds

In [ ]:
representation = "cnn"
npy_col = "CNN_SegEmb_npy"
experiments = [
    {"name": "GRU_LogMel", "model_name": "GRU", "kwargs": {}},
    {"name": "TRANS_LogMel", "model_name": "TRANS", "kwargs": {"pooling_trans": "mean"}}
]
for seed in [42, 123, 2026]:
    for exp in experiments:
        print(f"Processing {exp['name']} Regression | seed={seed}")
        results_reg = [
            rf.reg_lopoRNN(
                p, df_cnn, "Patient_ID", npy_col, target, exp["model_name"],
                representation=representation, num_epochs=num_epochs, BS=BS,
                seed=seed, **exp["kwargs"]
            )
            for p in unique_patients
        ]
        result_name = f"{exp['name']}_seed{seed}"
        df_results = pd.DataFrame(results_reg)
        df_results["Seed"] = seed
        df_results["Experiment"] = exp["name"]
        df_results.to_excel(f"/workspace/app/planilhas/TaCoLa/Seeds/REG_{result_name}.xlsx", index=False)
        print(f"{result_name} Done")

In [ ]:
elapsed_time = time.perf_counter() - start_time

print("\n---- Experiment finished ----")
print(f"Total execution time: {elapsed_time:.2f} seconds")
print(f"Total execution time: {elapsed_time / 60:.2f} minutes")